# Routing pattern

In the week 1 lab 2 we looked at a kind of Orchestration workflow where a question was sent to multiple LLMs and then a LLM decided upon the winning answer.

In this exersice we will be looking at the Router pattern using an LLM to decide which model is best fitted to answer the question.

In [128]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import random

In [129]:
load_dotenv(override=True)

True

In this example we will be using gpt-5.4 though OpenAI as the router. This model will decide what model will be handling the question.

In [130]:

router_llm_model = "gpt-5.4"
router_service = OpenAI()

Lets create some methods to ease our objective

In [131]:
question_models = {}
def add_question_model_service(url: str | None, api_key: str | None, model: str):
    if model in question_models.keys():
        return
    if url is None:
        question_models[model] = OpenAI()
        return
    if api_key is None:
        question_models[model] = OpenAI(base_url = url)
        return
    question_models[model] = OpenAI(base_url=url, api_key=api_key)

def get_question_service(model: str):
    if model not in question_models.keys():
        return None
    return question_models[model]

def format_messages(prompt: str):
    return [{"role": "user", "content": prompt}]

def prompt_model(model: str, prompt: str, service: OpenAI):    
    messages = format_messages(prompt)
    response = service.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

def get_random_level():
    models = list(question_models.keys())
    return f"Level {random.randint(0, len(models))}"
    


Lets define all available models and services available to answer questions

In [132]:
models = {
    "gpt-5.4-nano": None,
    "claude-sonnet-4-6": {
        "api_key": os.getenv('ANTHROPIC_API_KEY'),
        "base_url": "https://api.anthropic.com/v1/"
    },
    "moonshotai/kimi-k2.6":{
        "api_key": os.getenv('OPENROUTER_API_KEY'),
        "base_url": "https://openrouter.ai/api/v1"
    },
    "gemma4:latest":{
        "api_key": None,
        "base_url": "http://localhost:11434/v1"
    },
    "llama3.2:latest":{
        "api_key": None,
        "base_url": "http://localhost:11434/v1"
    },
    "gpt-oss:latest":{
        "api_key": None,
        "base_url": "http://localhost:11434/v1"
    }
}

def register_services():
    for model in models.keys():
        if models[model] is None:
            add_question_model_service(None, None, model)
        elif models[model]["api_key"] is None:
            add_question_model_service(models[model]["base_url"], None, model)
        else:
            add_question_model_service(models[model]["base_url"], models[model]["api_key"], model)

Now lets start by ranking the models based on their ability to answer questions

In [133]:
ranking_levels = [
    "Level 0: very simple reasoning; direct relationships and basic explanations",
    "Level 1: simple reasoning requiring a small number of logical steps",
    "Level 2: moderate reasoning involving multiple related concepts",
    "Level 3: substantial reasoning involving analysis, trade-offs, causality, or synthesis",
    "Level 4: difficult reasoning requiring careful multi-step analysis, abstraction, or competing considerations"
]

def rank_models():
    models_string = ", ".join(models.keys())
    levels = ""
    for l in ranking_levels:
        levels = f"* {l}\n"
    ranking_prompt =f"""
    You are ranking LLMs by their relative reasoning capability.

    The available complexity levels are:

    {levels}

    Models to rank:

    {models_string}

    Rank all provided models from weakest to strongest reasoning capability and assign each model exactly one unique complexity level.

    The lowest complexity level must be assigned to the weakest model, and the highest complexity level must be assigned to the strongest model.

    Evaluate relative reasoning capability primarily based on:

    * multi-step reasoning,
    * causal analysis,
    * abstraction,
    * synthesis,
    * trade-off analysis,
    * handling competing considerations,
    * maintaining coherence across dependent reasoning steps.

    Do not rank models primarily based on:

    * factual knowledge breadth,
    * context window size,
    * coding ability alone,
    * tool use or browsing capability,
    * speed,
    * cost,
    * multimodal capabilities.

    Requirements:

    * Every model must be assigned exactly once.
    * Every complexity level must contain exactly one model.
    * No two models may share a complexity level.
    * Rank models relative to each other, even when their capabilities are similar.
    * Use the provided complexity levels as the authoritative definition of the scale.
    * Preserve model names exactly as provided.
    * The number of models must equal the number of complexity levels.

    Return only a valid JSON array of model names ordered by complexity level.

    The array index represents the complexity level:

    * Index 0 = Level 0
    * Index 1 = Level 1
    * Index 2 = Level 2
    * And so on.

    Example output:

    ["weakest-model", "next-model", "middle-model", "stronger-model", "strongest-model"]

    Do not include explanations, markdown, level numbers, objects, or any text outside the JSON array.

    """
    display(Markdown(ranking_prompt))
    results = prompt_model(router_llm_model, ranking_prompt, router_service)
    return json.loads(results)

In [134]:
register_services()

ranked_models = rank_models()

print(ranked_models)


    You are ranking LLMs by their relative reasoning capability.

    The available complexity levels are:

    * Level 4: difficult reasoning requiring careful multi-step analysis, abstraction, or competing considerations


    Models to rank:

    gpt-5.4-nano, claude-sonnet-4-6, moonshotai/kimi-k2.6, gemma4:latest, llama3.2:latest, gpt-oss:latest

    Rank all provided models from weakest to strongest reasoning capability and assign each model exactly one unique complexity level.

    The lowest complexity level must be assigned to the weakest model, and the highest complexity level must be assigned to the strongest model.

    Evaluate relative reasoning capability primarily based on:

    * multi-step reasoning,
    * causal analysis,
    * abstraction,
    * synthesis,
    * trade-off analysis,
    * handling competing considerations,
    * maintaining coherence across dependent reasoning steps.

    Do not rank models primarily based on:

    * factual knowledge breadth,
    * context window size,
    * coding ability alone,
    * tool use or browsing capability,
    * speed,
    * cost,
    * multimodal capabilities.

    Requirements:

    * Every model must be assigned exactly once.
    * Every complexity level must contain exactly one model.
    * No two models may share a complexity level.
    * Rank models relative to each other, even when their capabilities are similar.
    * Use the provided complexity levels as the authoritative definition of the scale.
    * Preserve model names exactly as provided.
    * The number of models must equal the number of complexity levels.

    Return only a valid JSON array of model names ordered by complexity level.

    The array index represents the complexity level:

    * Index 0 = Level 0
    * Index 1 = Level 1
    * Index 2 = Level 2
    * And so on.

    Example output:

    ["weakest-model", "next-model", "middle-model", "stronger-model", "strongest-model"]

    Do not include explanations, markdown, level numbers, objects, or any text outside the JSON array.

    

['llama3.2:latest', 'gemma4:latest', 'gpt-oss:latest', 'gpt-5.4-nano', 'moonshotai/kimi-k2.6', 'claude-sonnet-4-6']


We will in this example use the same model as the Router to generate the question to process. This question could have been a user input as well.

In [ ]:
def get_question():
    random_level = get_random_level()
    print(f"Random level selected: {random_level}")
    levels = ""
    for l in ranking_levels:
        levels = f"* {l}\n"
    initial_question_request = f"""
    You are generating a question for an LLM reasoning benchmark.

    The available complexity levels are:

    {levels}

    The randomly selected target level is:

    {random_level}

    Generate one standalone question whose reasoning difficulty matches the selected level as closely as possible.

    Use the provided complexity levels as the authoritative definition of difficulty. Do not reinterpret the scale.

    The question should primarily test reasoning ability. Calibrate its difficulty through factors such as:

    * number of reasoning steps,
    * dependency between reasoning steps,
    * number of concepts that must be combined,
    * causal analysis,
    * trade-offs,
    * abstraction,
    * synthesis,
    * competing considerations.

    Do not make the question more difficult merely by using obscure facts, specialized terminology, complicated wording, or requiring current information.

    Requirements:

    * The question must be answerable primarily through reasoning and general knowledge.
    * Avoid mathematical puzzles, riddles, trivia, and questions requiring web searches or current information.
    * The question must have a reasonably assessable answer rather than being purely subjective.
    * Do not intentionally make the question easier or harder than the selected level.
    * Include an instruction in the question that the answer should be succinct, preferably 1–3 sentences.
    * Do not mention the complexity level, benchmark, evaluation process, or these instructions.

    Output only the generated question.


    """
    return prompt_model(router_llm_model, initial_question_request, router_service)

In [136]:
def rank_question_complexity(question: str):
    levels = ""
    for l in ranking_levels:
        levels = f"* {l}\n"
    prompt = f"""
    You are evaluating the reasoning complexity of a question.

    The available complexity levels are:

    {levels}

    Determine which level best matches the minimum reasoning capability required to produce a strong, correct answer to the question.

    Use the provided levels as the authoritative definition of complexity. Do not invent additional levels or reinterpret the scale.

    Judge the question based primarily on:

    * number of reasoning steps required,
    * dependency between reasoning steps,
    * number of concepts that must be combined,
    * depth of analysis,
    * causal reasoning,
    * trade-offs,
    * abstraction,
    * synthesis,
    * competing considerations.

    Do not assign a higher level merely because:

    * the topic is technical or specialized,
    * the wording is long or complicated,
    * uncommon terminology is used,
    * factual knowledge is required.

    A short question may require substantial reasoning, while a long question may require very little reasoning.

    Choose the LOWEST level that adequately represents the reasoning needed to give a strong answer.

    Question:

    {question}

    Output only the numeric level.
    """
    return prompt_model(router_llm_model, prompt, router_service)

In [137]:
def route_question(complexity: int, question: str):
    model = ranked_models[complexity]
    print(model)
    qa_service = get_question_service(model)
    return model, prompt_model(model, question, qa_service)

In [138]:
question = get_question()

complexity = int(rank_question_complexity(question))
print(complexity)

(model, answer) = route_question(complexity, question)

print(f"""The question \"{question}\" was ranked with a complexity of {complexity}.
Therefore the model \"{model}\" was used which came up with the following answer:
{answer}""")

4
moonshotai/kimi-k2.6
The question "A city is considering two policies to reduce traffic and air pollution in its downtown: Policy A would charge drivers a fee to enter the area during peak hours, while Policy B would ban most private cars entirely during those same hours but greatly expand bus service. Assume both policies would reduce car traffic substantially, but the city has limited enforcement capacity, many low-income workers commute from areas with weak transit access, and local businesses depend partly on customers who travel unpredictably. Which policy is more likely to produce better overall outcomes in the next 2–3 years, and why? Answer succinctly in 1–3 sentences." was ranked with a complexity of 4.
Therefore the model "moonshotai/kimi-k2.6" was used which came up with the following answer:
Policy A is likely to yield better outcomes over the next 2–3 years: a congestion fee preserves automobile access for essential trips, which protects low-income workers in poorly serv